In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

DATA_PATHS = [
    Path("../data/raw/laptops_cleaned.csv"),
    Path("data/raw/laptops_cleaned.csv"),
]
# Try to find the data file in the specified paths
data_path = next((p for p in DATA_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find laptops_cleaned.csv in the expected locations.")

df = pd.read_csv(data_path)

target = "price"
# Define the feature set (ex the target variable)
feature_set_a = [
    "brand",
    "device_category",
    "rating",
    "cpu_brand",
    "cpu_family",
    "cpu_core_count",
    "cpu_thread_count",
    "ram_gb",
    "storage_gb",
    "display_width_px",
    "display_height_px",
    "display_size_inch",
    "gpu_brand",
    "gpu_type",
    "gpu_vram_gb",
    "os_name",
    "warranty_years",
]
# Check for missing columns in the DataFrame
missing_columns = sorted(set(feature_set_a + [target]) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")
# Prepare feature matrix X and target vector y
X = df[feature_set_a].copy()
y = df[target].copy()
# Print the shapes of X and y, and the feature columns
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

In [ ]:
# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# Check for missing values in the feature matrix X
X.isnull().sum().sort_values(ascending=False)

In [ ]:
# Define preprocessing pipelines for numeric and categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
# Combine preprocessing steps into ColumnTransformers for linear and tree-based models
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_linear, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)
# The tree based preprocessor doesnt include scaling for numeric features
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)
# Print preprocessors to verify structure
print("Preprocessors created successfully.")

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Apply the linear preprocessor to the training and testing data
X_train_linear = preprocessor_linear.fit_transform(X_train)
X_test_linear = preprocessor_linear.transform(X_test)

# Check on the shapes after transforming
print("Transformed train shape:", X_train_linear.shape)
print("Transformed test shape :", X_test_linear.shape)

In [ ]:
# Get feature names after transformation for linear preprocessor
feature_names_linear = preprocessor_linear.get_feature_names_out()
pd.Series(feature_names_linear, name="transformed_feature").head(30)

In [ ]:
# Get feature names after transformation for tree based preprocessor
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

In [ ]:
# Define paths for saving metrics and figures
METRICS_DIR = Path("../outputs/metrics")
FIGURES_DIR = Path("../outputs/figures")

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Metrics path:", METRICS_DIR.resolve())
print("Figures path:", FIGURES_DIR.resolve())

In [ ]:
# Define models with their respective pipelines
models = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    "Ridge": Pipeline(steps=[
        ("preprocessor", preprocessor_linear),
        ("model", Ridge(alpha=1.0))
    ])
}

print(models)

In [ ]:
# Function to evaluate a regression model using cross validation and test set performance
def evaluate_regressor(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    cv,
    model_name
):
    scoring = {
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
    
# Perform cross validatin and compute metrics
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    y_pred = fitted_model.predict(X_test)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

# Compile results into a dictionary
    result = {
        "model": model_name,
        "cv_rmse_mean": -cv_results["test_rmse"].mean(),
        "cv_rmse_std": cv_results["test_rmse"].std(),
        "cv_mae_mean": -cv_results["test_mae"].mean(),
        "cv_mae_std": cv_results["test_mae"].std(),
        "cv_r2_mean": cv_results["test_r2"].mean(),
        "cv_r2_std": cv_results["test_r2"].std(),
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    }

    return result, fitted_model, y_pred

In [ ]:
# Define cross validation strategy
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv)

In [ ]:
# Evaluate each model and store results, fitted models, and test predictions 
results = []
fitted_models = {}
test_predictions = {}

# Loop through each model, evaluate, and store results
for model_name, model in models.items():
    result, fitted_model, y_pred = evaluate_regressor(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        cv=cv,
        model_name=model_name
    )

    results.append(result)
    fitted_models[model_name] = fitted_model
    test_predictions[model_name] = y_pred

# Create DataFrame to compare model performance
model_comparison_v1 = (
    pd.DataFrame(results)
    .sort_values(by="test_rmse", ascending=True)
    .reset_index(drop=True)
)

model_comparison_v1

In [ ]:
# Display the model comparison df with formatting for better readability
display(
    model_comparison_v1.style.format({
        "cv_rmse_mean": "{:.2f}",
        "cv_rmse_std": "{:.2f}",
        "cv_mae_mean": "{:.2f}",
        "cv_mae_std": "{:.2f}",
        "cv_r2_mean": "{:.4f}",
        "cv_r2_std": "{:.4f}",
        "test_rmse": "{:.2f}",
        "test_mae": "{:.2f}",
        "test_r2": "{:.4f}",
    })
)

In [ ]:
# Save the model comparison df to csv
comparison_path = METRICS_DIR / "model_comparison_v1.csv"
model_comparison_v1.to_csv(comparison_path, index=False)

print(f"Saved model comparison to: {comparison_path}")